# 08 — Problem exposures: modifications & collections scalability

**Plain English:** A prudent lender takes *early remedial action* on problem
exposures and watches whether restructuring actually works (APS 220 para 79;
APG 220 para 68). We use the SFLLD **modification flag** to find every loan that
was modified or payment-deferred, then ask the only question that matters: after
the modification, did it **cure** or **re-default**? We close with a one-paragraph
**collections-scalability** note — the crisis vintages already show how big an
arrears surge the book might have to absorb.

**One-line terms**
- **Modification / restructure** — a loss-mitigation change to a struggling loan (rate/term change, or payment deferral).
- **Re-default** — a modified loan that later reached 90+/default again; the test of whether remediation held.
- **Collections scalability** — whether the workout function could cope with a stress-driven multiple of today's arrears.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Modified / restructured-exposure outcomes by vintage -------------------
mod_view = m.modified_exposure_view(panel)
n_mod = int(mod_view["modified_loans"].sum())
print(f"{n_mod:,} loans ever modified / payment-deferred")
mod_view

5,254 loans ever modified / payment-deferred


,vintage,modified_loans,modified_exposure_upb,re_default_rate_pct,cure_rate_pct
0,2007,3079,43755532.0,51.1,44.1
1,2008,1712,27124608.0,44.1,50.6
2,2015,463,46233784.0,33.3,57.0


In [3]:
# Collections scalability: how big an arrears surge has the book seen? ----
# Monthly 30+DPD (arrears) rate per vintage; trough vs crisis peak = the surge
# multiple the collections function would need to absorb in a downturn.
arr = panel[panel.bucket.isin(m.ACTIVE_BUCKETS)].copy()
arr["is_arrears"] = arr["bucket"].isin(["30", "60", "90+"])
mo = (arr.groupby(["vintage", "period"])
      .agg(active=("loan_seq", "size"), arrears=("is_arrears", "sum")))
mo = mo[mo["active"] >= 100]
mo["arrears_pct"] = 100 * mo["arrears"] / mo["active"]
# Baseline = the cohort's TYPICAL month (median), not the ramp-up zero — so the
# surge multiple answers "peak was how many times a normal month?".
scal = (mo.groupby("vintage")["arrears_pct"]
        .agg(typical_arrears_pct="median", peak_arrears_pct="max").reset_index())
scal["surge_multiple"] = (scal.peak_arrears_pct / scal.typical_arrears_pct).round(1)
scal[["typical_arrears_pct", "peak_arrears_pct"]] = scal[["typical_arrears_pct", "peak_arrears_pct"]].round(2)
scal.to_csv(m.OUT_TABLES / "08_collections_scalability.csv", index=False)
mod_view.to_csv(m.OUT_TABLES / "08_modified_exposure.csv", index=False)
print("saved modified-exposure + collections-scalability tables")
scal

saved modified-exposure + collections-scalability tables


,vintage,typical_arrears_pct,peak_arrears_pct,surge_multiple
0,2007,11.71,15.69,1.3
1,2008,8.11,12.41,1.5
2,2015,1.61,4.89,3.0


In [4]:
# Hardship / concession outcomes incl. ULTIMATE loss (APG 220 para 68) ---
# Beyond cure vs re-default, the guidance asks for ultimate loss on the concession
# book — here proxied by the share that reached an actual Default credit-event
# termination (charge-off / short-sale / REO).
hardship = m.hardship_summary(panel)
hardship.to_csv(m.OUT_TABLES / "08_hardship_summary.csv", index=False)
hardship

,vintage,concessions,concession_exposure_upb,cure_rate_pct,re_default_rate_pct,ultimate_loss_rate_pct
0,2007,3079,43755532.0,44.1,51.1,9.8
1,2008,1712,27124608.0,50.6,44.1,6.7
2,2015,463,46233784.0,57.0,33.3,0.2


In [5]:
# Trend in NEW concession requests by year & product (APG 220 para 68) ---
# APRA expects the NUMBER of new requests by product type to be tracked. Note: the
# SFLLD records GRANTED concessions only, so an approval RATE is not observable here.
new_conc = m.new_concessions_by_year(panel)
new_conc.to_csv(m.OUT_TABLES / "08_new_concessions_by_year.csv", index=False)
new_conc

,year,new_concessions,owner_occupier,investor
0,2008,84,81,1
1,2009,339,320,15
2,2010,1368,1325,21
3,2011,952,931,10
4,2012,503,480,11
5,2013,559,519,27
6,2014,350,328,13
7,2015,227,202,18
8,2016,132,116,13
9,2017,119,103,13


In [6]:
# Unlikely-to-pay (UTP) overlay (APS 220 default definition) -------------
# Stage 3 here uses the 90-DPD / credit-event backstop only. APRA's default is
# 90 DPD *or* unlikely-to-pay; as an illustrative UTP signal we flag ever-modified
# loans still performing (<90 DPD) and on book. Production would fold qualitative
# UTP triggers into the default definition; here it is an extra early-warning count.
latest = panel.sort_values(["loan_seq", "mob"]).groupby("loan_seq").tail(1)
active = latest[latest.zb_code.isna()].copy()
utp_tbl = pd.DataFrame([m.utp_overlay(panel, active)])
utp_tbl.to_csv(m.OUT_TABLES / "08_utp_overlay.csv", index=False)
print("UTP overlay (illustrative early-warning):")
utp_tbl

UTP overlay (illustrative early-warning):


,utp_candidate_loans,utp_candidate_exposure_upb,utp_candidate_share_pct
0,639,110407044.0,6.361
